In [1]:
!pip install -q "datasets<3.0" transformers peft bitsandbytes accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
from huggingface_hub import login
login()  # will prompt for your token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Done! Model loaded in 4-bit quantization.")
print(f"Memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading tokenizer...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Done! Model loaded in 4-bit quantization.
Memory used: 6.1 GB


In [4]:
import re


def fix_column_names(sql: str, columns: list) -> str:
    """
    Wrap column name references in backticks, handling two common model mistakes:

    1. Direct unquoted: model writes  Nationality  → should be `Nationality`
    2. Underscore variant: model writes  NHL_team   → should be `NHL team`
       (the base model often converts spaces/dashes to underscores)
    """
    for col in sorted(columns, key=len, reverse=True):
        if not col:
            continue
        quoted = f'`{col}`'

        # Build the underscore variant: "NHL team" → "NHL_team", "Singles W–L" → "Singles_W_L"
        underscore_variant = re.sub(r'[\s\-–]+', '_', col)

        # Try both the original name and the underscore variant
        variants = [col]
        if underscore_variant.lower() != col.lower():
            variants.append(underscore_variant)

        for variant in variants:
            # Already correctly quoted — nothing to do
            if re.search(re.escape(quoted), sql, re.IGNORECASE):
                break
            if re.search(re.escape(variant), sql, re.IGNORECASE):
                sql = re.sub(re.escape(variant), quoted, sql, flags=re.IGNORECASE)
                break  # Fixed — stop trying variants for this column

    return sql


def generate_sql(question: str, columns: list, types: list, max_new_tokens: int = 128) -> str:
    """Generate SQL for a given question and table schema."""
    col_defs = ", ".join(f"{name} ({dtype})" for name, dtype in zip(columns, types))

    # Use the same prompt format as the training data (format_prompt in prepare_dataset.py)
    prompt = (
        f"### Input:\n"
        f"Columns: {col_defs}\n\n"
        f"Question: {question}\n\n"
        f"### SQL:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Remove markdown code blocks if present
    if "```" in response:
        lines = response.split('\n')
        response = ' '.join(l for l in lines if not l.strip().startswith('```')).strip()

    # Extract the first SELECT statement (stop at semicolon or newline)
    for line in response.split('\n'):
        line = line.strip()
        if line.upper().startswith('SELECT'):
            line = line.split(';')[0].strip()
            # Truncate any "### Answer:" / "### Note:" hallucination the model appended
            if '###' in line:
                line = line[:line.index('###')].strip()
            # Normalize table name to "table"
            line = re.sub(r'\bFROM\s+\w+\b', 'FROM table', line, flags=re.IGNORECASE)
            # Fix unquoted / underscore column names
            line = fix_column_names(line, columns)
            return line

    return response.split('\n')[0].split(';')[0].strip()


# Test on a single example
from datasets import load_dataset
ds = load_dataset("Salesforce/wikisql", trust_remote_code=True)
example = ds['test'][0]

predicted = generate_sql(
    example['question'],
    example['table']['header'],
    example['table']['types']
)

print(f"Question:  {example['question']}")
print(f"Predicted: {predicted}")
print(f"Gold:      {example['sql']['human_readable']}")

Generating test split:   0%|          | 0/15878 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8421 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/56355 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question:  What is terrence ross' nationality
Predicted: SELECT `Nationality` FROM table WHERE `Player` = 'Terrence Ross'
Gold:      SELECT Nationality FROM table WHERE Player = Terrence Ross


In [5]:
from tqdm.notebook import tqdm

examples = list(ds['test'].select(range(500)))
predictions = []

for example in tqdm(examples):
    pred = generate_sql(
        example['question'],
        example['table']['header'],
        example['table']['types']
    )
    predictions.append(pred)

print(f"Generated {len(predictions)} predictions")
print("\nSample predictions:")
for i in range(3):
    print(f"  {i+1}. {predictions[i]}")

  0%|          | 0/500 [00:00<?, ?it/s]

Generated 500 predictions

Sample predictions:
  1. SELECT `Nationality` FROM table WHERE `Player` = 'Terrence Ross'
  2. SELECT * FROM table WHERE `School/Club Team` = 'Toronto' AND `Years in Toronto` = '1995-96'
  3. SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '2003-06'


In [6]:
import sqlite3

def build_db(table):
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()
    headers = table["header"]
    types = table["types"]
    type_map = {"text": "TEXT", "number": "REAL", "real": "REAL"}
    col_defs = ", ".join(f'`{col}` {type_map.get(t, "TEXT")}' for col, t in zip(headers, types))
    cursor.execute(f"CREATE TABLE data ({col_defs})")
    for row in table["rows"]:
        converted = []
        for val, col_type in zip(row, types):
            if col_type in ("real", "number"):
                try:
                    converted.append(float(str(val).replace(",", "")))
                except:
                    converted.append(val)
            else:
                converted.append(val)
        placeholders = ", ".join(["?"] * len(converted))
        cursor.execute(f"INSERT INTO data VALUES ({placeholders})", converted)
    conn.commit()
    return conn

def execute_sql(table, sql):
    sql = sql.replace("FROM table", "FROM data")
    try:
        conn = build_db(table)
        cursor = conn.cursor()
        cursor.execute(sql)
        results = cursor.fetchall()
        conn.close()
        return results, None
    except Exception as e:
        return [], str(e)

def normalize_result(result):
    return sorted([str(row) for row in result])

def build_sql_string(sql, columns, types):
    AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
    COND_OPS = ["=", ">", "<"]
    agg = AGG_OPS[sql["agg"]]
    sel_col = columns[sql["sel"]]
    select_clause = f"SELECT {agg}(`{sel_col}`)" if agg else f"SELECT `{sel_col}`"
    where_clauses = []
    for col_idx, op_idx, cond in zip(
        sql["conds"]["column_index"],
        sql["conds"]["operator_index"],
        sql["conds"]["condition"],
    ):
        col = columns[col_idx]
        op = COND_OPS[op_idx]
        col_type = types[col_idx]
        if col_type in ("real", "number"):
            cleaned = str(cond).replace(",", "")
            try:
                float(cleaned)
                where_clauses.append(f"`{col}` {op} {cleaned}")
            except:
                where_clauses.append(f"`{col}` {op} '{str(cond).replace(chr(39), chr(39)*2)}'")
        else:
            escaped = str(cond).replace("'", "''")
            where_clauses.append(f"`{col}` {op} '{escaped}'")
    where_clause = " WHERE " + " AND ".join(where_clauses) if where_clauses else ""
    return select_clause + " FROM table" + where_clause

# Evaluate
total = len(predictions)
correct = 0
syntax_errors = 0
errors = []

for pred, example in zip(predictions, examples):
    table = example["table"]
    gold_sql = build_sql_string(example["sql"], table["header"], table["types"])
    pred_results, pred_error = execute_sql(table, pred)
    gold_results, _ = execute_sql(table, gold_sql)
    if pred_error:
        syntax_errors += 1
        errors.append({"pred": pred, "error": pred_error})
    elif normalize_result(pred_results) == normalize_result(gold_results):
        correct += 1

print(f"=== Base Model Results (n={total}) ===")
print(f"Execution accuracy: {correct/total:.1%}")
print(f"Syntax error rate:  {syntax_errors/total:.1%}")
print(f"Correct:            {correct}/{total}")

# Show a sample of remaining errors to understand what's still failing
if errors:
    print(f"\nSample syntax errors still occurring (first 3):")
    for e in errors[:3]:
        print(f"  SQL:   {e['pred']}")
        print(f"  Error: {e['error']}")
        print()

=== Base Model Results (n=500) ===
Execution accuracy: 37.2%
Syntax error rate:  21.8%
Correct:            186/500

Sample syntax errors still occurring (first 3):
  SQL:   SELECT Ship FROM (   SELECT Ship, `Launched`, ROW_NUMBER() OVER (PARTITION BY Ship ORDER BY `Launched`) AS RowNum   FROM table ) AS SubQuery WHERE `Launched` = 'September 1, 1964' AND RowNum = 1
  Error: no such column: Ship

  SQL:   SELECT Rank FROM table WHERE `Wrestler` = 'Bryan Danielson'
  Error: no such column: Rank

  SQL:   SELECT Rank FROM table WHERE `Wrestler` = 'Go Shiozaki'
  Error: no such column: Rank



In [7]:
import json

base_model_results = {
    "model": "meta-llama/Meta-Llama-3-8B-Instruct",
    "dataset": "wikisql",
    "split": "test",
    "n_examples": total,
    "execution_accuracy": correct / total,
    "syntax_error_rate": syntax_errors / total,
    "correct": correct,
    "syntax_errors": syntax_errors,
    "predictions": predictions  # save predictions for later analysis
}

with open("base_model_results.json", "w") as f:
    json.dump(base_model_results, f, indent=2)

print("Saved base_model_results.json")

Saved base_model_results.json
